# Try ReFactX with Rules

Generate a Nemo Datalog rule with Qwen, inspect the token mask, and run the result against an ownership graph. The demo source is the checked-out branch's `ReFactX-with-Rules/` directory (`nemo_rule.py` plus `stage1`–`stage5`); the notebook imports it directly, so run it from the ReFactX checkout. The default graph and all eight expected result sets are written directly in this notebook, so no CSV download or pre-existing dataset is required.

The second half walks through the mechanism explicitly: a character-level `RuleConstraint`, a `RuleTokenIndex`, and a `RuleLogitsProcessor` passed straight to `model.generate(..., logits_processor=...)`, with a `TextStreamer` so you watch the rule appear token by token. The `generate(...)` helper used by the benchmark is the same pipeline with bookkeeping added.

The notebook uses **Qwen/Qwen3.5-9B**, the active model in [Try ReFactX v3](https://github.com/rpo19/ReFactX/blob/69e23205125c44235de50842f351b89880e93093/notebooks/try_refactx_v3.ipynb). To reproduce the [server protocol](https://github.com/rpo19/ReFactX/blob/codex/nemo-rule-generation-milestone/ReFactX-with-Rules/MILESTONE_NEMO_RULE_GENERATION.md), choose the full synthetic graph and **Qwen/Qwen3.5-4B** on CPU float32.

Use a Python 3.12 Jupyter kernel and run the cells in order. Package installation and the first download of Qwen's tokenizer and weights require connectivity or a local cache. The Nemo-only cells can run before downloading any model files.

## Install the runtime

The installation runs in this kernel. Nemo is the Datalog engine from TU Dresden.

In [1]:
%pip install nemo-python==0.10.1 pandas==2.3.0 Faker==37.4.0 "torch>=2.7,<3" transformers==5.3.0 "accelerate>=1,<2"


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


If the install changes packages already imported in this kernel, restart the kernel and rerun from the top. The next cell reports which versions are active.

In [2]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from nmo_python import NemoEngine, load_string
from transformers import AutoTokenizer

# This notebook runs against the checked-out branch: the Nemo demo lives in
# ReFactX-with-Rules/ (nemo_rule.py + stage1..stage5), not in an embedded snapshot.
def _find_rules_dir():
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for rel in (Path("ReFactX-with-Rules"), Path("ReFactX") / "ReFactX-with-Rules"):
            candidate = base / rel
            if (candidate / "stage5" / "eval_rules.py").is_file():
                return candidate
    raise RuntimeError(
        "ReFactX-with-Rules/ not found; run this notebook from the ReFactX "
        "checkout (or set REFACTX_RULES_DIR)."
    )

_env_dir = os.environ.get("REFACTX_RULES_DIR")
RULES_DIR = Path(_env_dir).resolve() if _env_dir else _find_rules_dir()
for path in (RULES_DIR, RULES_DIR / "stage3", RULES_DIR / "stage4", RULES_DIR / "stage5"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eval_rules import (
    ABLATIONS, CHALLENGE_ABLATIONS, Eval, build_constraint, generate,
    load_model, make_prompt, prompt_ids, task_schema,
)
from kg_gen import make_entities, make_triples
from logits_processor import RuleDecodingError, RuleLogitsProcessor, RuleTokenIndex
from rule_constraint import RuleConstraint, build_trie
from rule_harness import _norm
from tasks import (
    CHALLENGE_TASKS, TASKS, compute_challenge_gold, compute_gold,
)

for package in ("nemo-python", "torch", "transformers"):
    print(f"{package} {version(package)}")
print("rules dir:", RULES_DIR)

nemo-python 0.10.1
torch 2.11.0
transformers 5.3.0
rules dir: /workspace/notebooks/ReFactX-with-Rules/ReFactX/ReFactX-with-Rules


## Run a tiny Nemo program

Read the rule from right to left. Each body atom must match a fact. Repeating `?y` connects the two links through the same company. The two stake variables can take different values. The head produces the output pair and `@output` asks Nemo to compute it.

In [3]:
toy = """
OWN("A", "B", 60) .
OWN("B", "C", 40) .

owner_of_owner(?x, ?z) :-
    OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .

@output owner_of_owner .
""".strip()

engine = NemoEngine(load_string(toy))
engine.reason()
print(toy)
print("Derived tuples", list(engine.result("owner_of_owner")))

OWN("A", "B", 60) .
OWN("B", "C", 40) .

owner_of_owner(?x, ?z) :-
    OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .

@output owner_of_owner .
Derived tuples [['"A"', '"C"']]


## Ownership graph and expected tuples

The small graph contains different stake sizes, repeated stakes, tied largest owners, and the real task names used by the quoted-entity tests. Both the graph and the expected outputs are visible in the next cell. Gold tuples are listed in the same argument order as each rule head.

Set `USE_FULL_SYNTHETIC_GRAPH = True` to regenerate the exact 100,000-entity synthetic benchmark in memory using the repo's `kg_gen` generator. Full-graph reference sets are calculated independently from the regenerated triples with pandas.

In [4]:
USE_FULL_SYNTHETIC_GRAPH = False

NAMED_OWNER = "Ronaldo Tomaselli"
NAMED_COMPANY = "Bellocchio-Peruzzi Group"

# Columns are subject, predicate, object, value.
INLINE_TRIPLES = [
    ("A", "OWN", "B", "60"),
    ("A", "OWN", "C", "20"),
    ("B", "OWN", "C", "40"),
    ("B", "OWN", NAMED_COMPANY, "45"),
    (NAMED_OWNER, "OWN", "C", "40"),
    (NAMED_OWNER, "OWN", NAMED_COMPANY, "45"),
    (NAMED_OWNER, "ROLE", "B", "director"),
    ("B", "REACHABLE", "C", ""),
]
INLINE_GOLD = {
    "owner_of_owner": {("A", "C"), ("A", NAMED_COMPANY)},
    "owner_count": {("B", 1), ("C", 3), (NAMED_COMPANY, 2)},
    "top_stake": {("B", 60), ("C", 40), (NAMED_COMPANY, 45)},
    "total_owned": {("A", 80), ("B", 85), (NAMED_OWNER, 85)},
    "majority_owner": {
        ("B", "A"),
        ("C", "B"), ("C", NAMED_OWNER),
        (NAMED_COMPANY, "B"), (NAMED_COMPANY, NAMED_OWNER),
    },
    "direct_majority_stake": {("A", "B")},
    "named_owner_stakes": {("C", 40), (NAMED_COMPANY, 45)},
    "named_company_owners": {("B", 45), (NAMED_OWNER, 45)},
}

if USE_FULL_SYNTHETIC_GRAPH:
    entities = make_entities()
    triples = make_triples(entities)
    golds = {**compute_gold(triples, write=False), **compute_challenge_gold(triples)}
    data_label = "Full synthetic graph"
else:
    triples = pd.DataFrame(INLINE_TRIPLES, columns=["subject", "predicate", "object", "value"])
    entities = pd.DataFrame({"entity": sorted(set(triples.subject) | set(triples.object))})
    golds = INLINE_GOLD
    data_label = "Inline teaching graph"

print(f"{data_label}: {len(entities):,} entities, {len(triples):,} base triples")
display(triples.head(8))
display(pd.DataFrame([
    {"task": name, "gold_rows": len(rows), "example_tuple": min(rows, key=repr)}
    for name, rows in golds.items()
]).set_index("task"))
evaluators = {}  # Drop the previous evaluator when switching graph modes.

Inline teaching graph: 5 entities, 8 base triples


,subject,predicate,object,value
0,A,OWN,B,60
1,A,OWN,C,20
2,B,OWN,C,40
3,B,OWN,Bellocchio-Peruzzi Group,45
4,Ronaldo Tomaselli,OWN,C,40
5,Ronaldo Tomaselli,OWN,Bellocchio-Peruzzi Group,45
6,Ronaldo Tomaselli,ROLE,B,director
7,B,REACHABLE,C,


,gold_rows,example_tuple
task,,
owner_of_owner,2,"(A, Bellocchio-Peruzzi Group)"
owner_count,3,"(B, 1)"
top_stake,3,"(B, 60)"
total_owned,3,"(A, 80)"
majority_owner,5,"(B, A)"
direct_majority_stake,1,"(A, B)"
named_owner_stakes,2,"(Bellocchio-Peruzzi Group, 45)"
named_company_owners,2,"(B, 45)"


In [5]:
# Each task retains its own prompt, target head, gold rule and any context rules.
suites = {
    "pilot": {"tasks": TASKS, "arms": ABLATIONS},
    "challenge": {"tasks": CHALLENGE_TASKS, "arms": CHALLENGE_ABLATIONS},
}
task_by_id = {task["id"]: task for suite in suites.values() for task in suite["tasks"]}
suite_by_task = {task["id"]: name for name, suite in suites.items() for task in suite["tasks"]}



def evaluator(task_id):
    if "graph" not in evaluators:
        evaluators["graph"] = Eval(triples=triples, entity_names=entities.entity, golds=golds)
    return evaluators["graph"]


display(pd.DataFrame([
    {"suite": suite_by_task[task_id], "task": task_id, "instruction": task["instruction"]}
    for task_id, task in task_by_id.items()
]).set_index(["suite", "task"]))

instruction
suite     task                                                                    
pilot     owner_of_owner         Write a Datalog rule defining the relation own...
          owner_count            Write a Datalog rule defining the relation own...
          top_stake              Write a Datalog rule defining the relation top...
          total_owned            Write a Datalog rule defining the relation tot...
          majority_owner         Write a Datalog rule defining the relation maj...
challenge direct_majority_stake  Define direct_majority_stake(owner, company) f...
          named_owner_stakes     Define named_owner_stakes(company, stake_perce...
          named_company_owners   Define named_company_owners(owner, stake_perce...

## Explain the actual two-hop miss from the pilot

At seed 43, both constrained arms reused the stake variable in the two body atoms. That extra equality excluded chains with different stakes. The default graph demonstrates the same behavior on two visible chains. In full-graph mode, the candidate misses 164,636 pairs and adds none, matching the saved server diagnostic.

In [6]:
# Check every supplied gold rule before generating any new rule.
for task_id, definition in task_by_id.items():
    reference = evaluator(task_id).score(definition["gold_rule"], definition)
    if reference["outcome"] != "correct":
        raise AssertionError((task_id, reference))

task = task_by_id["owner_of_owner"]
ev = evaluator(task["id"])
pilot_miss = "owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?s), OWN(?y, ?z, ?s) ."
examples = {"reference": task["gold_rule"], "pilot miss at seed 43": pilot_miss}
rows = []
for label, rule in examples.items():
    print(label, rule, sep="\n")
    rows.append({"example": label, **ev.score(rule, task)})

display(pd.DataFrame(rows)[["example", "outcome", "rows", "missing", "extra"]])

reference
owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .
pilot miss at seed 43
owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?s), OWN(?y, ?z, ?s) .


,example,outcome,rows,missing,extra
0,reference,correct,2,0,0
1,pilot miss at seed 43,valid_wrong,0,2,0


## Load the model

`load_model` chooses the device (`mps` → `cuda` → `cpu`), loads the dtype, and falls back to an
image-text model class if the causal-LM class rejects the checkpoint. Everything below uses batch
size 1.

In [7]:
#MODEL_ID = "Qwen/Qwen3.5-9B"            # active model in try_refactx_v3.ipynb
MODEL_ID = "Qwen/Qwen3.5-4B"              # server benchmark model
#MODEL_ID = "Qwen/Qwen3.5-2B"              # server benchmark model
#MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # small local smoke test

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
DEVICE = 'cpu'
DTYPE = "float32" if DEVICE == "cpu" else (
    "bfloat16" if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else "float16"
)
DEVICE_MAP = {"": DEVICE}   # set to "auto" to let Accelerate place the weights.
SEED = 42
TASK_ID = "owner_of_owner"
ARM = "full"    # "unconstrained" | "grammar_distractors" | "full" (challenge: "entity_distractors")

task = task_by_id[TASK_ID]



print(f"{MODEL_ID} | {DEVICE} | {DTYPE} | task={TASK_ID} | arm={ARM}")

Qwen/Qwen3.5-4B | cpu | float32 | task=owner_of_owner | arm=full


In [8]:
model, tokenizer, loaded_device = load_model(
    MODEL_ID, dtype=DTYPE, device=DEVICE, device_map=DEVICE_MAP,
)
print(f"loaded {MODEL_ID} on {loaded_device}, vocab={len(tokenizer):,}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded Qwen/Qwen3.5-4B on cpu, vocab=248,077


## How the constrained decoding works

Three small objects, none of which know about Qwen:

1. **`RuleConstraint`** (`stage4/rule_constraint.py`) — a character-level state machine built from
   the head, its arity, the allowed body predicates (with arities), and the entity trie. It exposes
   `initial`, `step(state, ch)`, `next_chars(state)`, `accepting(state)`, `can_complete(state)`, and
   `consume(state, text)`. It decides which **characters** may come next.
2. **`RuleTokenIndex`** (`stage4/logits_processor.py`) — decodes every vocabulary id once and groups
   ids by their **first character** (`by_first[ch] -> [token_id, ...]`).
3. **`RuleLogitsProcessor`** — the bridge. At each generation step it decodes the generated suffix,
   `consume`s it through the constraint to get the current state, keeps only token ids whose spelling
   is a viable continuation **from which completion is still possible**, and sets every other logit to
   `-inf`. Once the state is `accepting`, only `eos` is allowed, so generation stops on a complete rule.

Because the mask is applied token by token, streaming works exactly as in plain `generate`.

### 1. The grammar as a character-level machine

We build the entity trie from the graph above and hand it — together with the task's schema — to
`build_constraint`. With `ARM="full"` this returns a `RuleConstraint` for `owner_of_owner/2`.

In [9]:
# Ground the quoted constants the model may emit with the graph's entities.
entity_names = sorted(entities.entity)
entity_trie = build_trie(entity_names)
constraint = build_constraint(task, ARM, entity_trie)
print("constraint:", type(constraint).__name__, "| entities:", len(entity_names))

if constraint is not None:
    state = constraint.initial
    print("next chars at start:", sorted(constraint.next_chars(state)))
    for ch in "owner_of_owner(":
        state = constraint.step(state, ch)
        assert state, f"constraint died on {ch!r}"
    print("after 'owner_of_owner(', next chars:", sorted(constraint.next_chars(state)))

    gold = task["gold_rule"]
    done = constraint.consume(constraint.initial, gold)
    print("gold rule accepted:", constraint.accepting(done),
          "| can still complete:", constraint.can_complete(done))

constraint: RuleConstraint | entities: 5
next chars at start: ['o']
after 'owner_of_owner(', next chars: ['"', '#', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '?']
gold rule accepted: True | can still complete: True


### 2. Characters → token ids

The tokenizer splits text into tokens, so the processor needs each token's spelling. `RuleTokenIndex`
decodes the whole vocabulary once and buckets ids by first character.

In [10]:
token_index = RuleTokenIndex(tokenizer)
n_spellings = sum(len(ids) for ids in token_index.by_first.values())
print(f"{len(tokenizer):,} vocabulary ids; {n_spellings:,} have a non-empty spelling")
print("tokens starting with 'o':", [token_index.strings[i] for i in token_index.by_first["o"][:12]])

248,077 vocabulary ids; 248,068 have a non-empty spelling
tokens starting with 'o': ['o', 'on', 'or', 'ou', 'om', 'ol', 'od', 'ot', 'ow', 'ort', 'out', 'our']


### 3. One masking step

Here is the processor applied once to a dummy score vector at position 0, showing which tokens survive.
(Tokenizing the chat prompt is `prompt_ids`; `generation_start` is its length, so the mask ignores the
user turn.)

In [11]:
prompt = make_prompt(task)
input_ids = prompt_ids(tokenizer, prompt).to(model.device)
generation_start = input_ids.shape[-1]
print("prompt tokens:", generation_start)

if constraint is not None:
    probe = RuleLogitsProcessor(token_index, constraint, generation_start)
    scores = torch.zeros(len(tokenizer), device=model.device)
    masked = probe(input_ids, scores)
    allowed = torch.isfinite(masked).nonzero(as_tuple=True)[0].tolist()
    print(f"step 0: {len(allowed):,} of {len(tokenizer):,} tokens allowed")
    print("e.g.", [token_index.strings[i] for i in allowed[:15]])
else:
    print("ARM is 'unconstrained': there is no mask.")

prompt tokens: 244
step 0: 4 of 248,077 tokens allowed
e.g. ['o', 'ow', 'own', 'owner']


## `model.generate` with the logits processor, streamed live

This is the whole mechanism in one call: pass a `RuleLogitsProcessor` in a `LogitsProcessorList`.
A `TextStreamer` prints each decoded token as it is produced, so the rule appears incrementally.

Hugging Face may print a one-line warning that the checkpoint's default `temperature`/`top_p`/`top_k` are ignored because `do_sample=False`. That is about the model's `generation_config`, not the mask; the constrained decoding still applies at every step.

In [12]:
from transformers import LogitsProcessorList, TextStreamer

processor = (RuleLogitsProcessor(token_index, constraint, generation_start)
             if constraint is not None else None)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

torch.manual_seed(SEED)
with torch.inference_mode():
    out = model.generate(
        input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=160,
        do_sample=False,
        num_beams=1,
        num_return_sequences=1,
        logits_processor=LogitsProcessorList([processor] if processor is not None else []),
        streamer=streamer,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None
                      else tokenizer.eos_token_id),
        eos_token_id=tokenizer.eos_token_id,
    )

new_ids = out[0, generation_start:]
rule_text = tokenizer.decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print("\n\n=== decoded rule ===")
print(rule_text)
if processor is not None:
    print(f"tokens: {len(new_ids)} | mask steps: {processor.mask_steps} | mask time: {processor.mask_s:.1f}s")

owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .


=== decoded rule ===
owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .
tokens: 32 | mask steps: 32 | mask time: 10.1s


### Score the rule against gold tuples

`Eval.score` validates the rule shape, runs it in Nemo, and compares the output tuples to the
independently computed gold set.

In [15]:
def show_candidate(task_id, text, limit=5):
    result = evaluator(task_id).score(text, task_by_id[task_id], include_tuples=True)
    display(pd.Series({k: v for k, v in result.items() if k != "tuples"}))
    if "tuples" in result:
        actual = result["tuples"]
        gold = {tuple(_norm(v) for v in row) for row in evaluator(task_id).golds[task_id]}
        for label, examples in (("Result", actual), ("Missing", gold - actual), ("Extra", actual - gold)):
            if examples:
                preview = sorted(examples, key=repr)[:limit]
                print(f"{label} ({len(examples):,} total)")
                display(pd.DataFrame(preview, columns=["arg1", "arg2"]))
    return result

scored = show_candidate(TASK_ID, rule_text)

nemo_s                                                0.033052
rows                                                         2
outcome                                                correct
missing                                                      0
extra                                                        0
rule_text    owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN...
dtype: object

Result (2 total)


,arg1,arg2
0,A,Bellocchio-Peruzzi Group
1,A,C


## The same thing, packaged

`generate(...)` in `stage5/eval_rules.py` is exactly the call above plus it reports the termination
reason, token count, mask time, and whether decoding hit a grammar dead end. The benchmark in the last
cell uses this helper, so the walkthrough and the numbers are the same code path.

In [16]:
generation = generate(
    model, tokenizer, prompt, constraint, token_index,
    seed=SEED, do_sample=False, max_new_tokens=160,
)
print(generation["raw_text"])
display(pd.Series({key: value for key, value in generation.items() if key != "raw_text"}))
if generation["generation_error"]:
    print("generation_error:", generation["generation_error"])
else:
    show_candidate(TASK_ID, generation["raw_text"])

owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN(?y, ?z, ?q) .


n_tokens                    32
termination                eos
gen_s               101.305099
generation_error              
mask_steps                  32
mask_s               10.112469
dtype: object

nemo_s                                                0.031232
rows                                                         2
outcome                                                correct
missing                                                      0
extra                                                        0
rule_text    owner_of_owner(?x, ?z) :- OWN(?x, ?y, ?p), OWN...
dtype: object

Result (2 total)


,arg1,arg2
0,A,Bellocchio-Peruzzi Group
1,A,C


## Try a count and a quoted entity

Use the same model and evaluator for a second task. The helper returns the raw response and score record, including the mask time and any error. Each example is a separate generation so its rule and outcome can be inspected immediately.

In [17]:
def ask_rule(task_id, *, arm="full", seed=42, sample=False, instruction=None):
    task = task_by_id[task_id]
    active_suite = suites[suite_by_task[task_id]]
    if arm not in active_suite["arms"]:
        raise ValueError(f"Use one of {active_suite['arms']} for this task.")
    if instruction is not None:
        task = dict(task, instruction=instruction)
    row, _ = evaluator(task_id).run_one(
        model, tokenizer, task, arm, {"do_sample": sample, "max_new_tokens": 160}, seed,
    )
    print(row["raw_text"])
    display(pd.Series({key: value for key, value in row.items()
                       if key not in ("raw_text", "rule_text")}))
    return row

In [18]:
# The reference counts distinct owners per company using a head aggregate.
count_result = ask_rule("owner_count")

owner_count(?y, #count(?x, ?z)) :- OWN(?x, ?y, ?z) .


task           owner_count
ablation              full
seed                    42
n_tokens                26
termination            eos
gen_s            91.271445
mask_steps              26
mask_s           12.563225
nemo_s            0.031699
rows                     3
outcome            correct
missing                  0
extra                    0
wall_s           92.897075
dtype: object

In [19]:
# The company name must be emitted as a quoted constant. Try entity_distractors too.
named_result = ask_rule("named_company_owners")

named_company_owners(?x, ?y) :- OWN(?x, "Bellocchio-Peruzzi Group", ?y) .


task           named_company_owners
ablation                       full
seed                             42
n_tokens                         29
termination                     eos
gen_s                     93.548615
mask_steps                       29
mask_s                     6.864235
nemo_s                     0.033669
rows                              2
outcome                     correct
missing                           0
extra                             0
wall_s                    93.599097
dtype: object

## Try your own wording or hand-written rule

Choose a task, edit its instruction, and rerun this cell. The score still uses that task's original gold relation, so keep the requested output columns and meaning if you want to compare scores. Run `show_candidate(task_id, rule_text)` to evaluate a hand-written rule without calling Qwen.

In [20]:
MY_TASK = "majority_owner"
MY_INSTRUCTION = (
    "Define majority_owner(company, owner) for all direct owners tied for "
    "the largest stake in their company. Use the supplied top_stake relation."
)
MY_ARM = "full"        # Also try unconstrained or grammar_distractors.
MY_SEED = 43
MY_SAMPLE = False

my_result = ask_rule(
    MY_TASK, arm=MY_ARM, seed=MY_SEED, sample=MY_SAMPLE, instruction=MY_INSTRUCTION,
)

majority_owner(?x, ?y) :- OWN(?y, ?x, ?p), top_stake(?x, ?p), top_stake(?x, ?p), CONNECTED(?x, ?y) .


task           majority_owner
ablation                 full
seed                       43
n_tokens                   48
termination               eos
gen_s              140.158408
mask_steps                 48
mask_s              13.556194
nemo_s               0.034446
rows                        0
outcome           valid_wrong
missing                     5
extra                       0
wall_s             140.208688
dtype: object

## Optional benchmark and CSV

Set `USE_FULL_SYNTHETIC_GRAPH = True` and switch the model to `Qwen/Qwen3.5-4B` and use CPU float32 to match the server configuration. Choose `pilot` for five tasks × three arms or `challenge` for three tasks × four arms. With three samples per task and arm, the pilot has 45 rows and the challenge has 36.

Set `BENCHMARK_TASK_IDS = ["owner_of_owner"]` and `N_SAMPLES = 1` for a short, paired comparison. All arms get the same prompt and seeds. `grammar_distractors` admits extra predicate names and `entity_distractors` admits a near-match alias on named tasks. `full` uses the KG catalogs. `valid_wrong` means the rule executed but its result differed from gold. The cell saves complete rows as a CSV after the run.

In [ ]:
from datetime import datetime

RUN_BENCHMARK = False
BENCHMARK_SUITE = "pilot"
BENCHMARK_TASK_IDS = None  # None uses all tasks in the selected suite.
N_SAMPLES = 3
BENCHMARK_SEED = 42
BENCHMARK_SAMPLE = True

if RUN_BENCHMARK:
    suite = suites[BENCHMARK_SUITE]
    selected_tasks = suite["tasks"]
    if BENCHMARK_TASK_IDS is not None:
        requested = set(BENCHMARK_TASK_IDS)
        selected_tasks = [task for task in selected_tasks if task["id"] in requested]
        if not requested or requested != {task["id"] for task in selected_tasks}:
            raise ValueError(f"Choose task IDs from the {BENCHMARK_SUITE} suite.")
    bench = evaluator(selected_tasks[0]["id"])
    results = bench.run(
        model, tokenizer, tasks=selected_tasks, ablations=suite["arms"],
        n_samples=N_SAMPLES, seed=BENCHMARK_SEED, cfg={"do_sample": BENCHMARK_SAMPLE},
        verbose=True,
    )
    display(bench.summary(results))
    display(results.assign(correct=results["outcome"].eq("correct")).pivot_table(
        index="task", columns="ablation", values="correct", aggfunc="mean",
    ).mul(100).round(1))
    results_dir = Path.cwd() / "nemo_notebook_results"
    results_dir.mkdir(parents=True, exist_ok=True)
    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    out = results_dir / f"rules_notebook_{'full_graph' if USE_FULL_SYNTHETIC_GRAPH else 'inline'}_{BENCHMARK_SUITE}_{run_stamp}.csv"
    results.to_csv(out, index=False)
    print(out)
else:
    print("Set RUN_BENCHMARK to True to run the selected suite.")

For the headless server commands and the shared-question QA comparison, see the [project README](https://github.com/rpo19/ReFactX/tree/codex/nemo-rule-generation-milestone/ReFactX-with-Rules).